<a href="https://colab.research.google.com/github/thuongngo050902/INFERENCE-THEISIS/blob/main/Inference_MAT_%3D_THESIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Wed May 13 03:37:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -lh /content/drive/MyDrive/THESIS/checkpoints

total 1.9G
-rw------- 1 root root 646M May 10 03:14 network-snapshot-000072.pkl
-rw------- 1 root root 631M May 10 06:19 Places_512_FullData.pkl
-rw------- 1 root root 619M May 10 03:18 resume_phase1_from_finetune_plus_loss.pkl


In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

%cd /content
!git clone -b codex/phase3-thesis-ui-demo https://{GITHUB_TOKEN}@github.com/thuongngo050902/THESIS.git
%cd /content/THESIS

/content
Cloning into 'THESIS'...
remote: Enumerating objects: 390, done.
remote: Counting objects: 100% (390/390), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 390 (delta 186), reused 354 (delta 151), pack-reused 0 (from 0)
Receiving objects: 100% (390/390), 18.99 MiB | 16.57 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/content/THESIS


In [ ]:
!git pull

Already up to date.


In [ ]:
!ls
!ls demo_ui

colab_inference_api.py	docs		   metrics	      test_sets
collab			evaluatoin	   networks	      torch_utils
datasets		figures		   README.md	      training
dataset_tool.py		generate_image.py  requirements.txt   train.py
demo_ui			legacy.py	   smoke_test_ffl.py
DEPLOY_SERVER.md	LICENSE		   streamlit_app.py
dnnlib			losses		   test_1
inference_adapter.py  __init__.py


In [ ]:
!pip install -q fastapi uvicorn python-multipart pillow opencv-python pyngrok nest_asyncio requests

In [ ]:
!pip install -q streamlit streamlit-drawable-canvas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 84.0 MB/s eta 0:00:00


In [ ]:
!pip install -q ninja imageio imageio-ffmpeg scipy tqdm click psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.8 MB/s eta 0:00:00


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 23.0 MB/s eta 0:00:00
  Attempting uninstall: imageio-ffmpeg
    Found existing installation: imageio-ffmpeg 0.6.0
    Uninstalling imageio-ffmpeg-0.6.0:
      Successfully uninstalled imageio-ffmpeg-0.6.0


## Test import

In [ ]:
from pathlib import Path

FINAL_CKPT = "/content/drive/MyDrive/THESIS/checkpoints/network-snapshot-000072.pkl"
STAGE1_CKPT = "/content/drive/MyDrive/THESIS/checkpoints/resume_phase1_from_finetune_plus_loss.pkl"
BASELINE_CKPT = "/content/drive/MyDrive/THESIS/checkpoints/Places_512_FullData.pkl"

for p in [FINAL_CKPT, STAGE1_CKPT, BASELINE_CKPT]:
    print(Path(p).exists(), p)

True /content/drive/MyDrive/THESIS/checkpoints/network-snapshot-000072.pkl
True /content/drive/MyDrive/THESIS/checkpoints/resume_phase1_from_finetune_plus_loss.pkl
True /content/drive/MyDrive/THESIS/checkpoints/Places_512_FullData.pkl


In [ ]:
%cd /content/THESIS

import sys
sys.path.insert(0, "/content/THESIS")

from demo_ui.inference_adapter import run_generator_on_inputs, ensure_binary_mask
print("Imported inference adapter successfully")

/content/THESIS
Imported inference adapter successfully


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Fast API server

#### Patch code for final output

In [ ]:
from pathlib import Path

p = Path("/content/THESIS/demo_ui/inference_adapter.py")
s = p.read_text()

old = "generator, _generator_key, _network_data = load_generator_for_inference(network_pkl, device)"
new = "loaded = load_generator_for_inference(network_pkl, device)\n        generator = loaded[0] if isinstance(loaded, tuple) else loaded"

if old not in s:
    print("Old line not found. Show nearby lines:")
    for i, line in enumerate(s.splitlines(), start=1):
        if "load_generator_for_inference" in line:
            print(i, line)
else:
    s = s.replace(old, new)
    p.write_text(s)
    print("patched", p)

Old line not found. Show nearby lines:
13 from generate_image import load_generator_for_inference
114         loaded = load_generator_for_inference(


#### Patch code for stage 1 output

In [ ]:
from pathlib import Path

p = Path("/content/THESIS/colab_inference_api.py")
s = p.read_text()

old = """run_generator_on_inputs(
                checkpoint_path,
                input_image,
                mask_image,
                device,
            )"""

new = """run_generator_on_inputs(
                checkpoint_path,
                input_image,
                mask_image,
                device,
                allow_missing_params=(checkpoint == "stage1"),
            )"""

if old not in s:
    print("Target block not found. Nearby lines:\n")

    lines = s.splitlines()
    for i, line in enumerate(lines, start=1):
        if "run_generator_on_inputs(" in line:
            start = max(0, i - 3)
            end = min(len(lines), i + 10)

            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")
            print("\n---\n")
else:
    s = s.replace(old, new)
    p.write_text(s)
    print("✅ patched colab api")

Target block not found. Nearby lines:

104: 
105:         with torch.inference_mode():
106:             output_image = run_generator_on_inputs(
107:                 checkpoint_path,
108:                 input_image,
109:                 mask_image,
110:                 device,
111:                 allow_missing_params=allows_missing_checkpoint_params(checkpoint),
112:             )
113: 
114:         elapsed = time.time() - started
115: 
116:         if response_format == "image":

---



In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run(
        "colab_inference_api:app",
        host="0.0.0.0",
        port=8000,
        reload=False,
    )

thread = threading.Thread(target=run_api)
thread.start()

#### Test locally server

In [ ]:
import requests

r = requests.get("http://127.0.0.1:8000/health")
print(r.status_code)
print(r.json())

INFO:     127.0.0.1:44118 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'ok', 'cuda_available': True, 'gpu_name': 'Tesla T4', 'checkpoints': {'stage1': {'path': '/content/drive/MyDrive/THESIS/checkpoints/resume_phase1_from_finetune_plus_loss.pkl', 'exists': True}, 'final': {'path': '/content/drive/MyDrive/THESIS/checkpoints/network-snapshot-000072.pkl', 'exists': True}, 'mat_baseline': {'path': '/content/drive/MyDrive/THESIS/checkpoints/Places_512_FullData.pkl', 'exists': True}}}


## Expose API with ngrok

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

public_tunnel = ngrok.connect(8000, "http")
public_url = public_tunnel.public_url

print("Public API URL:", public_url)
print("Health:", public_url + "/health")

Public API URL: https://salaried-easter-epileptic.ngrok-free.dev
Health: https://salaried-easter-epileptic.ngrok-free.dev/health
